# Day 9 — Practical MLOps Showcase

Practical Colab-only work:
- train/save/load model
- reusable sklearn pipeline
- generate FastAPI app
- generate requirements.txt + Dockerfile
- MLflow experiment tracking
- basic model versioning
- data drift + prediction drift checks

Docker execution itself is intentionally left for local VS Code/terminal.


## Install

In [ ]:
!pip -q install mlflow fastapi uvicorn joblib scikit-learn scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.6/144.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

## Imports

In [ ]:
import os, json, joblib, zipfile
import numpy as np, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import ks_2samp
import mlflow, mlflow.sklearn

## Train a real model

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))

Accuracy: 0.9824561403508771
Precision: 0.9861111111111112
Recall: 0.9861111111111112
F1: 0.9861111111111112


## Save + reload model

In [ ]:
joblib.dump(pipeline, "breast_cancer_pipeline.pkl")
loaded = joblib.load("breast_cancer_pipeline.pkl")

sample = X_test.iloc[[0]]
print("Prediction:", loaded.predict(sample)[0])
print("Confidence:", loaded.predict_proba(sample)[0].max())

Prediction: 0
Confidence: 0.9999999411175814


## Create FastAPI app

In [ ]:
fastapi_code = '''
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()
model = joblib.load("breast_cancer_pipeline.pkl")

class InputData(BaseModel):
    values: list[float]

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(data: InputData):
    X = pd.DataFrame([data.values], columns=model.feature_names_in_)
    pred = model.predict(X)[0]
    proba = model.predict_proba(X)[0].max()
    return {"prediction": int(pred), "confidence": float(proba)}
'''

open("main.py","w").write(fastapi_code)
print(open("main.py").read())


import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()
model = joblib.load("breast_cancer_pipeline.pkl")

class InputData(BaseModel):
    values: list[float]

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(data: InputData):
    X = pd.DataFrame([data.values], columns=model.feature_names_in_)
    pred = model.predict(X)[0]
    proba = model.predict_proba(X)[0].max()
    return {"prediction": int(pred), "confidence": float(proba)}



## Create requirements.txt

In [ ]:
requirements = '''fastapi
uvicorn
scikit-learn
pandas
numpy
joblib
mlflow
'''
open("requirements.txt","w").write(requirements)
print(open("requirements.txt").read())

fastapi
uvicorn
scikit-learn
pandas
numpy
joblib
mlflow



## Create Dockerfile

In [ ]:
dockerfile = '''FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["uvicorn","main:app","--host","0.0.0.0","--port","8000"]
'''
open("Dockerfile","w").write(dockerfile)
print(open("Dockerfile").read())

FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["uvicorn","main:app","--host","0.0.0.0","--port","8000"]



## MLflow run 1

In [ ]:
mlflow.set_experiment("day9-mlops-demo")

with mlflow.start_run(run_name="logreg-v1"):
    model1 = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=1.0, max_iter=2000))
    ])
    model1.fit(X_train, y_train)
    p1 = model1.predict(X_test)

    mlflow.log_param("C", 1.0)
    mlflow.log_metric("accuracy", accuracy_score(y_test,p1))
    mlflow.log_metric("f1", f1_score(y_test,p1))
    mlflow.sklearn.log_model(model1, "model")

print("Run 1 logged")

2026/09/17 11:52:49 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/17 11:52:49 INFO mlflow.store.db.utils: Updating database tables
2026/09/17 11:52:56 INFO mlflow.tracking.fluent: Experiment with name 'day9-mlops-demo' does not exist. Creating a new experiment.
2026/09/17 11:52:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run 1 logged


## MLflow run 2

In [ ]:
with mlflow.start_run(run_name="logreg-v2"):
    model2 = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=0.1, max_iter=2000))
    ])
    model2.fit(X_train, y_train)
    p2 = model2.predict(X_test)

    mlflow.log_param("C", 0.1)
    mlflow.log_metric("accuracy", accuracy_score(y_test,p2))
    mlflow.log_metric("f1", f1_score(y_test,p2))
    mlflow.sklearn.log_model(model2, "model")

print("Run 2 logged")

2026/09/17 11:53:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run 2 logged


## Simple model versioning

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(model1, "models/model_v1.pkl")
joblib.dump(model2, "models/model_v2.pkl")
print(os.listdir("models"))

['model_v1.pkl', 'model_v2.pkl']


## Simulate production drift

In [ ]:
production_X = X_test.copy()
production_X["mean radius"] *= 1.15
production_X["mean texture"] *= 1.10

for feature in ["mean radius","mean texture","mean perimeter"]:
    stat, p = ks_2samp(X_train[feature], production_X[feature])
    print(feature, "| drift:", p < 0.05, "| p:", round(float(p),6))

mean radius | drift: True | p: 0.0
mean texture | drift: True | p: 4.9e-05
mean perimeter | drift: False | p: 0.538772


## Prediction drift

In [ ]:
ref_rate = pipeline.predict(X_train).mean()
prod_rate = pipeline.predict(production_X).mean()

print("Reference positive rate:", round(float(ref_rate),4))
print("Production positive rate:", round(float(prod_rate),4))
print("Shift:", round(abs(float(ref_rate-prod_rate)),4))

Reference positive rate: 0.633
Production positive rate: 0.5965
Shift: 0.0365


## Create project bundle

In [ ]:
metrics = {
    "accuracy": float(accuracy_score(y_test,pred)),
    "precision": float(precision_score(y_test,pred)),
    "recall": float(recall_score(y_test,pred)),
    "f1": float(f1_score(y_test,pred))
}
open("metrics.json","w").write(json.dumps(metrics, indent=2))

monitoring = {
    "reference_positive_rate": float(ref_rate),
    "production_positive_rate": float(prod_rate),
    "prediction_rate_shift": float(abs(ref_rate-prod_rate))
}
open("monitoring.json","w").write(json.dumps(monitoring, indent=2))

with zipfile.ZipFile("day9_mlops_showcase.zip","w") as z:
    for f in [
        "breast_cancer_pipeline.pkl","main.py","requirements.txt",
        "Dockerfile","metrics.json","monitoring.json"
    ]:
        z.write(f)

print("Created day9_mlops_showcase.zip")

Created day9_mlops_showcase.zip


## Run locally after downloading

In [ ]:
print(r'''
pip install -r requirements.txt
uvicorn main:app --reload

Open:
http://127.0.0.1:8000/docs

Docker:
docker build -t ml-api .
docker run -p 8000:8000 ml-api
''')


pip install -r requirements.txt
uvicorn main:app --reload

Open:
http://127.0.0.1:8000/docs

Docker:
docker build -t ml-api .
docker run -p 8000:8000 ml-api

